# Decoder Block


In [1]:
from pathlib import Path
import torch
import torch.nn as nn

LECTURE_NOTE_TITLE = 'Decoder Block'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')

class DecoderBlock(nn.Module):
    def __init__(self, d_model=8, n_heads=2, d_ff=32):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
        attn_out, weights = self.attn(x, x, x, attn_mask=mask, need_weights=True, average_attn_weights=False)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ffn(x))
        return x, weights


Lecture note: Decoder Block


## Decoder block preserves shape


In [2]:
torch.manual_seed(0)
x = torch.randn(1, 4, 8)
block = DecoderBlock()
y, weights = block(x)
print(x.shape, '->', y.shape)


torch.Size([1, 4, 8]) -> torch.Size([1, 4, 8])


## Causal mask blocks future positions


In [3]:
print(weights[0, 0])


tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6458, 0.3542, 0.0000, 0.0000],
        [0.4409, 0.1868, 0.3723, 0.0000],
        [0.3506, 0.2058, 0.1848, 0.2588]], grad_fn=<SelectBackward0>)


## Compare with unmasked encoder-style attention


In [4]:
encoder_attn = nn.MultiheadAttention(8, 2, batch_first=True)
_, bidirectional = encoder_attn(x, x, x, need_weights=True, average_attn_weights=False)
print('decoder head 0:')
print(weights[0, 0])
print('encoder head 0:')
print(bidirectional[0, 0])


decoder head 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6458, 0.3542, 0.0000, 0.0000],
        [0.4409, 0.1868, 0.3723, 0.0000],
        [0.3506, 0.2058, 0.1848, 0.2588]], grad_fn=<SelectBackward0>)
encoder head 0:
tensor([[0.1528, 0.2216, 0.4302, 0.1955],
        [0.2527, 0.2959, 0.1677, 0.2837],
        [0.3282, 0.2638, 0.1474, 0.2605],
        [0.1921, 0.2943, 0.2452, 0.2684]], grad_fn=<SelectBackward0>)


## Decoder blocks drive autoregressive GPT models


A decoder block combines: causal self-attention + FFN + residuals + normalization
It keeps the sequence length and d_model width unchanged so many blocks can stack cleanly.


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py
